# Otimização — capítulo 4: Otimização com restrições

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_2_barrier/ex04_3_2_barrier.ipynb)

Corra a célula **Preparação** e depois o exemplo (no Colab: *Runtime → Run all*). No fim, o exemplo compara os resultados com os slides (*confere com os slides: sim*). Os slides usam vírgula decimal; aqui usa-se o ponto. Caderno gerado a partir de `ex04_3_2_barrier.py` por `notebooks/make_notebooks.py`.

*Code comments and printed messages are in Portuguese.*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 4.3.2 Método da função de barreira / penalização interior

Exemplo [`ex04_3_2_barrier.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_2_barrier/ex04_3_2_barrier.py) · funções da UC: [`barrier_log.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_2_barrier/barrier_log.py), [`nelder_mead_comp.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/common/nelder_mead_comp.py), [`penalty_exterior.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/04_constrained_optimization/04_3_1_exterior_penalty/penalty_exterior.py), [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py)

```text
Reproduz os exemplos do deck 4.3.2 (barreira logarítmica).

1. «Ideia»: min x^2 s.a. x - 1 >= 0; x_R = (1 + sqrt(1+2R))/2: 1.366, 1.048, 1.005
   (R = 1, 0.1, 0.01).
2. «O exemplo nas contas» e «As KKT reaparecem --- perturbadas»: min x1^2 + x2^2
   s.a. x1 + x2 - 1 >= 0, R = 1, 0.1, 0.01, 0.001: x_R, g(x_R), ||x_R - x*|| = O(R), u_R.
3. Barreira inversa R/g, R = 0.01: x_R = (0.548; 0.548), contra (0.5050; 0.5050) da log.
4. Restrição inativa: min (x1-1)^2 + (x2-1)^2 s.a. 9 - x1^2 - x2^2 >= 0:
   R = 1 -> (0.882; 0.882), R = 0.1 -> (0.986; 0.986); u_R = R/g -> 0.
5. A chamada do slide «No computador» (tolcomp = 1e-8, tolx = 1e-6, tmax = 12).
6. A comparação exterior vs. interior (slide «Exterior vs. interior»): barreira de
   (1,1) em 5 ciclos, 743 chamadas a P, 26 fora de X, n_f = 717; exterior de (0,0)
   em 6 ciclos, n_f = 844, e de (1,1), n_f = 864 (até ||x^(t) - x*|| <= 1e-4, sem a
   avaliação final). Usa penalty_exterior da pasta 04_3_1_exterior_penalty.

Minimizador interno: o Nelder–Mead da UC na versão do script da comparação
(nelder_mead_comp, tolx = 1e-10, tolf = 1e-12, kmax = 2000; ver o cabeçalho
de nelder_mead_comp.py e o README). As tabelas são uma única corrida com
arranque a quente (c = 0.1), lida em history.
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [2]:
import numpy as np

from barrier_log import barrier_log
from nelder_mead_comp import nelder_mead_comp
from penalty_exterior import penalty_exterior  # (4.3.1)


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


NM = lambda P, x: nelder_mead_comp(P, x, 1e-10, 1e-12, 2000)   # interno dos exemplos
ok = True
print("Otimização — deck 4.3.2: barreira logarítmica")

# ------------------------------------------------------------ 1. Ideia (1D)
r = barrier_log(lambda x: x[0]**2, lambda x: x[0] - 1, [2.0], 1, 0.1, 0, 0, 3, NM)
xR = r.history[1:, 5]
print("\n1. min x^2 s.a. x - 1 >= 0, de x = 2: x_R = (1 + sqrt(1+2R))/2")
for Ri, xi in zip(r.history[1:, 1], xR):
    print("   R = %5g: x_R = %.4f  (exato %.4f)" % (Ri, xi, (1 + np.sqrt(1 + 2 * Ri)) / 2))
c = [confere(xR, [1.366, 1.048, 1.005], 3)]
print("  x_R = 1.366, 1.048, 1.005: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 2. O exemplo nas contas
f = lambda x: x[0]**2 + x[1]**2
g = lambda x: x[0] + x[1] - 1
xs = np.array([0.5, 0.5])
r = barrier_log(f, g, [1, 1], 1, 0.1, 0, 0, 4, NM)
H = r.history[1:]
Rv = H[:, 1]; X = H[:, 5:7]; u = H[:, 7]
gv = X.sum(axis=1) - 1; err = np.linalg.norm(X - xs, axis=1)
print("\n2. min x1^2 + x2^2 s.a. x1 + x2 - 1 >= 0, de (1,1), R^(0) = 1, c = 0.1 (x* = (1/2,1/2), u* = 1)")
print("  %6s %20s %10s %12s %12s %8s %6s %6s"
      % ("R", "x_R", "g(x_R)", "||x_R-x*||", "||x_R-x*||/R", "u_R", "P", "fora"))
for i in range(4):
    print("  %6g   (%.4f; %.4f) %10.4f %12.5f %12.4f %8.4f %6d %6d"
          % (Rv[i], X[i, 0], X[i, 1], gv[i], err[i], err[i] / Rv[i], u[i], H[i, 2], H[i, 3]))
print("  erro O(R): ||x_R - x*||/R -> 1/sqrt 2 = %.4f;  u_R = R/g = (1 + sqrt(1+4R))/2 -> 1;"
      "  u_R g(x_R) = R" % (1 / np.sqrt(2)))
c = [confere(X, np.repeat([0.809, 0.5458, 0.5050, 0.5005], 2).reshape(4, 2),
             np.repeat([3, 4, 4, 4], 2).reshape(4, 2)),
     confere(gv, [0.618, 0.0916, 0.0099, 0.0010], [3, 4, 4, 4]),
     confere(err, [0.437, 0.0648, 0.0070, 0.00071], [3, 4, 4, 5]),
     confere(u, [1.618, 1.092, 1.010, 1.001], 3),
     all(gv > 0)]
print("  x_R: %s | g(x_R): %s | ||x_R - x*||: %s | u_R: %s | todos admissíveis: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. Barreira inversa
R = 0.01
Pinv = lambda x: np.inf if g(x) <= 0 else f(x) + R / g(x)
ri = nelder_mead_comp(Pinv, [1, 1], 1e-10, 1e-12, 2000)
print("\n3. Barreira inversa P = f + R/g, R = 0.01, de (1,1): x_R = (%.4f; %.4f), ||x_R - x*|| = %.4f"
      % (ri.x[0], ri.x[1], np.linalg.norm(ri.x - xs)))
print("   log (tabela acima), R = 0.01: x_R = (%.4f; %.4f), ||x_R - x*|| = %.4f  -> inversa O(sqrt R), log O(R)"
      % (X[2, 0], X[2, 1], err[2]))
c = [confere(ri.x, [0.548, 0.548], 3), confere(X[2], [0.505, 0.505], 3)]
print("  inversa (0.548; 0.548): %s | log (0.505; 0.505): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 4. Restrição inativa
r = barrier_log(lambda x: (x[0] - 1)**2 + (x[1] - 1)**2, lambda x: 9 - x[0]**2 - x[1]**2,
                [0, 0], 1, 0.1, 0, 0, 2, NM)
H = r.history[1:]
print("\n4. Restrição inativa: min (x1-1)^2 + (x2-1)^2 s.a. 9 - x1^2 - x2^2 >= 0, de (0,0) (x* = (1,1), u* = 0)")
for row in H:
    print("   R = %4g: x_R = (%.4f; %.4f), g = %.4f, u_R = R/g = %.4f"
          % (row[1], row[5], row[6], 9 - row[5]**2 - row[6]**2, row[7]))
c = [confere(H[:, 5:7], [[0.882, 0.882], [0.986, 0.986]], 3), bool(H[1, 7] < H[0, 7] < 1)]
print("  x_R = (0.882; 0.882), (0.986; 0.986): %s | u_R a descer para 0: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 5. Chamada do slide
print("\n5. Chamada do slide: barrier_log(f, g, [1, 1], 1, 0.1, 1e-8, 1e-6, 12)")
print("   (interno por omissão: nelder_mead_comp; o slide usa o fminsearch do MATLAB)")
r = barrier_log(f, g, [1, 1], 1, 0.1, 1e-8, 1e-6, 12, verbose=True)
nfc = r.history[1:, 2]
print("   nit = %d (R = %.3g), u = %.4f, chamadas a P por ciclo %d-%d, nfev = chamadas a P + 1 = %d"
      % (r.nit, r.R, r.u[0], nfc.min(), nfc.max(), r.nfev))
print("   fora de X: %d; n_f efetivo = %d (nfev é majorante de n_f); %s"
      % (r.nfora, r.nfev_efetivo, r.message))
print("   nota: R no ciclo 9 = 0.1^8 calculado como %.17g > 1e-8 = tolcomp: J*R <= tolcomp"
      % r.history[9, 1])
print("         só pode valer a partir do ciclo 10 (R ~ 1e-9), qualquer que seja o minimizador interno")
print("   slide (fminsearch do MATLAB): nit = 10 (R = 1e-9), u ~ 1; nfev ~ 140-190 por ciclo")
c = [abs(r.u[0] - 1) <= 0.01]
print("  u ~ 1 (a menos de 1 %%): %s | nit = 10: %s (informativo: depende do minimizador interno)"
      % (simnao(c)[0], "sim" if r.nit == 10 else "não"))
ok = ok and all(c)

# ------------------------------------------------------------ 6. Comparação
print("\n6. Comparação exterior vs. interior: o mesmo Nelder–Mead interno (tolerâncias 1e-10),")
print("   arranque a quente; para no 1.º ciclo com ||x^(t) - x*|| <= 1e-4 (lido em history)")


def ate(H, tol=1e-4):
    """1.º ciclo t (linha de history) com ||x^(t) - x*|| <= tol."""
    e = np.linalg.norm(H[1:, 5:7] - xs, axis=1)
    return int(np.argmax(e <= tol)) + 1


rb = barrier_log(f, g, [1, 1], 1, 0.1, 0, 0, 12, NM)
tb = ate(rb.history); Hb = rb.history
nPb = int(Hb[tb, 4]); forab = int(Hb[1:tb + 1, 3].sum())
print("   barreira de (1,1), R^(0) = 1, c = 0.1:")
for row in Hb[1:tb + 1]:
    print("     t = %d  R = %8.1e  x = (%.5f, %.5f)  ||x - x*|| = %.1e  P = %3d (fora %2d)  acum. = %d"
          % (row[0], row[1], row[5], row[6], np.linalg.norm(row[5:7] - xs), row[2], row[3], row[4]))
print("     -> %d ciclos, %d chamadas a P, %d fora de X, n_f = %d (sem a avaliação final)"
      % (tb, nPb, forab, nPb - forab))
ext = {}
for x0 in ((0.0, 0.0), (1.0, 1.0)):
    re = penalty_exterior(f, g, lambda x: [], list(x0), 0.1, 10, 0, 0, 12, NM)
    te = ate(re.history)
    ext[x0] = (te, int(re.history[te, 3]), int(np.median(re.history[1:te + 1, 2])))
print("   exterior de (0,0), R^(0) = 0.1, c = 10: %d ciclos, n_f = %d" % ext[(0.0, 0.0)][:2])
print("   exterior de (1,1), R^(0) = 0.1, c = 10: %d ciclos, n_f = %d" % ext[(1.0, 1.0)][:2])
medb = int(np.median(Hb[1:tb + 1, 2]))
print("\n                                                 exterior  barreira")
print("   ciclos exteriores t                          %9d %9d" % (ext[(0.0, 0.0)][0], tb))
print("   por ciclo (mediana; n_f | chamadas a P)      %9d %9d" % (ext[(0.0, 0.0)][2], medb))
print("   total até ||x^(t) - x*|| <= 1e-4             %9d %9d" % (ext[(0.0, 0.0)][1], nPb))
c = [(tb, nPb, forab, nPb - forab) == (5, 743, 26, 717),
     ext[(0.0, 0.0)][:2] == (6, 844), ext[(1.0, 1.0)][:2] == (6, 864),
     (ext[(0.0, 0.0)][2], medb) == (140, 146)]
print("  barreira 5 ciclos, 743 P, 26 fora, n_f = 717: %s | exterior (0,0) 6 ciclos, 844: %s |"
      " exterior (1,1) 6 ciclos, 864: %s | medianas 140 e 146: %s" % simnao(c))
ok = ok and all(c)

# informativo: o mesmo com o nelder_mead do deck 3.2.3 (contração exterior aceite com <=)
from nelder_mead import nelder_mead  # noqa: E402  (deck 3.2.3)
rb3 = barrier_log(f, g, [1, 1], 1, 0.1, 0, 0, 12, lambda P, x: nelder_mead(P, x, 1e-10, 1e-12, 2000))
t3 = ate(rb3.history); H3 = rb3.history
print("   informativo — com nelder_mead do deck 3.2.3 (aceita a contração exterior com <=):")
print("   barreira de (1,1): %d ciclos, %d chamadas a P, %d fora de X, n_f = %d"
      % (t3, H3[t3, 4], H3[1:t3 + 1, 3].sum(), H3[t3, 4] - H3[1:t3 + 1, 3].sum()))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 4.3.2: barreira logarítmica

1. min x^2 s.a. x - 1 >= 0, de x = 2: x_R = (1 + sqrt(1+2R))/2
   R =     1: x_R = 1.3660  (exato 1.3660)
   R =   0.1: x_R = 1.0477  (exato 1.0477)
   R =  0.01: x_R = 1.0050  (exato 1.0050)
  x_R = 1.366, 1.048, 1.005: sim

2. min x1^2 + x2^2 s.a. x1 + x2 - 1 >= 0, de (1,1), R^(0) = 1, c = 0.1 (x* = (1/2,1/2), u* = 1)
       R                  x_R     g(x_R)   ||x_R-x*|| ||x_R-x*||/R      u_R      P   fora
       1   (0.8090; 0.8090)     0.6180      0.43702       0.4370   1.6180    145      0
     0.1   (0.5458; 0.5458)     0.0916      0.06478       0.6478   1.0916    154      2
    0.01   (0.5050; 0.5050)     0.0099      0.00700       0.7002   1.0099    146      5
   0.001   (0.5005; 0.5005)     0.0010      0.00071       0.7064   1.0010    144      8
  erro O(R): ||x_R - x*||/R -> 1/sqrt 2 = 0.7071;  u_R = R/g = (1 + sqrt(1+4R))/2 -> 1;  u_R g(x_R) = R
  x_R: sim | g(x_R): sim | ||x_R - x*||: sim | u_R: sim | todos admissíveis: sim

3. 

   exterior de (0,0), R^(0) = 0.1, c = 10: 6 ciclos, n_f = 844
   exterior de (1,1), R^(0) = 0.1, c = 10: 6 ciclos, n_f = 864

                                                 exterior  barreira
   ciclos exteriores t                                  6         5
   por ciclo (mediana; n_f | chamadas a P)            140       146
   total até ||x^(t) - x*|| <= 1e-4                   844       743
  barreira 5 ciclos, 743 P, 26 fora, n_f = 717: sim | exterior (0,0) 6 ciclos, 844: sim | exterior (1,1) 6 ciclos, 864: sim | medianas 140 e 146: sim


   informativo — com nelder_mead do deck 3.2.3 (aceita a contração exterior com <=):
   barreira de (1,1): 5 ciclos, 739 chamadas a P, 26 fora de X, n_f = 713

confere com os slides: sim
